# 01. Data Preparation
## EmotionRadar | DM1590 KTH Final Project

## What this notebook does
- Loads GoEmotions dataset from HuggingFace (43K train / 5K val / 5K test)
- Maps 28 original emotion labels → 7 Ekman categories
- Produces multi-label binary encoding (7 columns of 0/1 per row)
- Saves 3 CSV files to data/processed/

## Input files
- data/processed/train.csv
- data/processed/val.csv
- data/processed/test.csv

## How to run
Pull latest from GitHub. Files are already there. Run cells top to bottom.

In [25]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [26]:
# Root of the repo
ROOT = Path("..") 

# Raw CSVs
RAW_DIR = ROOT / "data" / "raw"

# Processed output
PROCESSED_DIR = ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [27]:
from datasets import load_dataset

# Load GoEmotions from HuggingFace (aggregated, deduplicated, pre-split)
dataset = load_dataset("go_emotions", "simplified")

# Convert each split to a pandas DataFrame
train_raw = dataset["train"].to_pandas()
val_raw   = dataset["validation"].to_pandas()
test_raw  = dataset["test"].to_pandas()

print("Train:", train_raw.shape)
print("Val:  ", val_raw.shape)
print("Test: ", test_raw.shape)
print("Columns:", train_raw.columns.tolist())

Train: (43410, 3)
Val:   (5426, 3)
Test:  (5427, 3)
Columns: ['text', 'labels', 'id']


In [28]:
# Ekman taxonomy mapping (Demszky et al., 2020 — GoEmotions paper)
ekman_mapping = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude",
                 "love", "optimism", "relief", "pride", "admiration", 
                 "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"]
}

#print("Ekman categories:", list(ekman_mapping.keys()))

In [29]:
# Get the 28 label names in order (index 0-27)
label_names = dataset["train"].features["labels"].feature.names

# Build a lookup: emotion name → its integer index
label_to_idx = {name: idx for idx, name in enumerate(label_names)}

# Ekman mapping (same as before)
ekman_mapping = {
    "anger":    ["anger", "annoyance", "disapproval"],
    "disgust":  ["disgust"],
    "fear":     ["fear", "nervousness"],
    "joy":      ["joy", "amusement", "approval", "excitement", "gratitude",
                 "love", "optimism", "relief", "pride", "admiration",
                 "desire", "caring"],
    "sadness":  ["sadness", "disappointment", "embarrassment", "grief", "remorse"],
    "surprise": ["surprise", "realization", "confusion", "curiosity"],
    "neutral":  ["neutral"]
}

def build_ekman_df(split_df):
    df = pd.DataFrame()
    df["text"] = split_df["text"]
    
    for ekman_label, source_labels in ekman_mapping.items():
        # Get integer indices for this Ekman bucket
        source_indices = [label_to_idx[s] for s in source_labels]
        # A row gets 1 if ANY source index appears in its labels list
        df[ekman_label] = split_df["labels"].apply(
            lambda label_list: int(any(idx in label_list for idx in source_indices))
        )
    return df

train_df = build_ekman_df(train_raw)
val_df   = build_ekman_df(val_raw)
test_df  = build_ekman_df(test_raw)

print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

Train: (43410, 8)
Val:   (5426, 8)
Test:  (5427, 8)


In [31]:
train_df.to_csv(PROCESSED_DIR / "train.csv", index=False)
val_df.to_csv(PROCESSED_DIR  / "val.csv",   index=False)
test_df.to_csv(PROCESSED_DIR / "test.csv",  index=False)

print("Saved 3 splits to", PROCESSED_DIR)
print("Train:", train_df.shape)
print("Val:  ", val_df.shape)
print("Test: ", test_df.shape)

Saved 3 splits to ../data/processed
Train: (43410, 8)
Val:   (5426, 8)
Test:  (5427, 8)
